# Run Spark marine pipeline

Capstone Spark job (no dashboard app needed):

1. Fetch Open-Meteo marine conditions for default ports
2. Write Delta silver as a **Unity Catalog table** (avoids disabled public DBFS `/tmp`)
3. Sync ports, snapshots, and narrative docs into Lakebase

**Next:** `ingest_marine_embeddings.ipynb`, then Playground + MCP.

Attach **Serverless** or a cluster with Spark, then run the cells below.

In [ ]:
import importlib.util
from pathlib import Path

pipeline_path = Path.cwd() / "spark_marine_pipeline.py"
if not pipeline_path.exists():
    pipeline_path = Path.cwd() / "notebooks" / "spark_marine_pipeline.py"

spec = importlib.util.spec_from_file_location("spark_marine_pipeline", pipeline_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()
print("Pipeline finished")
print("Delta target:", getattr(mod, "_LAST_DELTA_TARGET", None))

In [ ]:
catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
schema = spark.sql("SELECT current_schema()").collect()[0][0]
table = f"{catalog}.{schema}.coastal_ops_marine_conditions"
print("Reading", table)
display(spark.table(table))